In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyPDFLoader(file_path="Map of AI.pdf")
documensts = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")
splits = text_splitter.split_documents(documents=documensts)

vectorstore = FAISS.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())
retriever = vectorstore.as_retriever()

C:\Users\Aditya\AppData\Local\Temp\ipykernel_24220\2889444762.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  vectorstore = FAISS.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())
C:\Users\Aditya\AppData\Local\Temp\ipykernel_24220\2889444762.py:7: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vectorstore = FAISS.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

message = """
Answer this question using the provided context only. If you dont know the answer, just say 'I dont know'
{input}
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("chat_history"),
    ("human", message)
])

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

chat_history = []

input_text = """Give me the policy names related to employee attendance, 
         punctuality and conflict of interest. 
         Give me only names as list"""

response = rag_chain.invoke({
    "chat_history": chat_history,
    "input": input_text
})

print(response["answer"])
# here we are not providing context because retriever will retrieve the context

I don't know.


In [11]:
response['answer']

"I don't know."

In [18]:
from langchain_core.messages import AIMessage, HumanMessage
chat_history = []
chat_history.extend(
    [
        HumanMessage(content=input_text),
        AIMessage(content=response["answer"])
    ]
)

response = rag_chain.invoke({
    "chat_history": chat_history,
    "input" : """Can you elaborate about those points
                 and give me response as bullet points in different lines"""
})

print(response['answer'])

# as it halusinates that is why we need a multi query retrival

I don't know.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever

contextualize_q_system_prompt = """Given above chat history and the below latest user question 
which might reference context in the chat history,
formulate a standalone question which can be understood
without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
Below is the latest question:

{input}
"""

# this is how we create conversational RAG as this prompt is interacting with llm as getting new prompt

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        # ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", f"""{contextualize_q_system_prompt}"""),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm,retriever, contextualize_q_prompt
)

In [21]:
result=history_aware_retriever.invoke(
{
    "chat_history":chat_history,
    "input" : """Can you elaborate about those points
    and give me response as bullet points in different lines"""
}
)

result

[Document(id='dca37917-36b6-4068-bc9b-02231b5b2fc4', metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Map of AI', 'source': 'Map of AI.pdf', 'total_pages': 66, 'page': 63, 'page_label': '64'}, page_content='P\nE\nO\nP\nL\nE\n● Educators – teach others how to use \nAI effectively and responsibly.'),
 Document(id='7f48929b-a305-4358-91f2-50324ca89f44', metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Map of AI', 'source': 'Map of AI.pdf', 'total_pages': 66, 'page': 26, 'page_label': '27'}, page_content='RESEARCH\nFOUNDATION\nPLATFORM\nUser Layer\nBUILDER\nAPPLICATION\nOPERATION\nDISTRIBUTION\nUSER\nFocus Areas\n● User Interaction\n● Personalization\n● Prompt Literacy\n● Trust & Ethics\n● Feedback Collection\nOutput\nUser data, preferences and \nbehavioral data'),
 Document(id='eae7ada1-9963-402d-a354-2198c676ee96', metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Map of AI', 'source': 'Map 

In [26]:
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Answer this question using the provided context only. If you dont know the answer, just say 'I dont know'"
    "\n\n"
    "context: {context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
[
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
contextual_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [23]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []
question = """Give me the policy names related to employee attendance , punctuality and conflict of interest.
Give me only names as list"""

print(question)
ai_msg_1 = contextual_rag_chain.invoke({"input": question, "chat_history": chat_history})
print(ai_msg_1['answer'])

Give me the policy names related to employee attendance , punctuality and conflict of interest.
Give me only names as list
I am sorry, but this document does not contain the answer to your question.


In [ ]:
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=ai_msg_1["answer"]),
    ]
)

second_question = "Can you elaborate about those policies and give me response as bullet points in different lines"
print(second_question)
ai_msg_2 = contextual_rag_chain.invoke({"input": second_question, "chat_history": chat_history})

print(ai_msg_2["answer"])

Can you elaborate about those points and give me response as bullet points in different lines
I am sorry, but this document does not contain the answer to your question.
